In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

""" Загрузка данных датасета """
digits = load_digits()

X = digits.data.astype(np.float32)
Y = digits.target

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

""" 64 -> 8 -> 10 """
rng = np.random.default_rng(1)

W1 = rng.normal(0, 0.01, (64, 8))
b1 = np.zeros(8)

W2 = rng.normal(0, 0.01, (8, 10))
b2 = np.zeros(10)

learning_rate = 0.001

def relu(x):
    return np.maximum(x, 0)

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

""" Обучение """
for epoch in range(20000):
    z1 = X_train @ W1 + b1

    h = relu(z1)

    z2 = h @ W2 + b2

    probs = softmax(z2)

    target = np.zeros_like(probs)
    target[np.arange(len(Y_train)), Y_train] = 1

    dz2 = (probs - target) / len(X_train)

    dW2 = h.T @ dz2
    db2 = np.sum(dz2, axis=0)

    dh = dz2 @ W2.T

    dz1 = dh * (z1 > 0)

    dW1 = X_train.T @ dz1
    db1 = np.sum(dz1, axis=0)

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

""" Проверка """
z1 = X_test @ W1 + b1
h = relu(z1)
z2 = h @ W2 + b2

pred = np.argmax(z2, axis=1)
accuracy = np.mean(pred == Y_test)

print("Accuracy:", accuracy)


In [ ]:
""" Квантизация """
W1_scale = np.max(np.abs(W1)) / 127.0
W1_int = np.round(W1 / W1_scale).astype(np.int8)
X_test_int = X_test.astype(np.int32)
b1_int = np.round(b1 / W1_scale).astype(np.int32)

""" Integer inference первого слоя """
z1_int = X_test_int @ W1_int.astype(np.int32) + b1_int
h_int32 = np.maximum(z1_int, 0)

""" Квантизация выхода первого слоя """
h_real_max = np.max(h_int32) * W1_scale
h_scale = h_real_max / 127.0

h_int = np.round(h_int32 * W1_scale / h_scale)
h_int = np.clip(h_int, 0, 127).astype(np.int8)

""" Второй слой """
W2_scale = np.max(np.abs(W2)) / 127.0
W2_int = np.round(W2 / W2_scale).astype(np.int8)
b2_int = np.round(b2 / (h_scale * W2_scale)).astype(np.int32)

""" Integer inference второго слоя """
z2_int = (h_int.astype(np.int32) @ W2_int.astype(np.int32) + b2_int)

""" Проверка """
pred_int = np.argmax(z2_int, axis=1)
accuracy_int = np.mean(pred_int == Y_test)

print("Float accuracy:", accuracy)
print("INT8 accuracy:", accuracy_int)

In [ ]:
""" Квантизация первого слоя """
W1_scale = np.max(np.abs(W1)) / 127.0
W1_int = np.round(W1 / W1_scale).astype(np.int8)
X_test_int = X_test.astype(np.int32)
b1_int = np.round(b1 / W1_scale).astype(np.int32)

""" Integer inference первого слоя """
z1_int = (X_test_int @ W1_int.astype(np.int32) + b1_int)
h_int32 = np.maximum(z1_int, 0)

""" Квантизация выхода первого слоя через сдвиг """
SHIFT = 7
h_int = (h_int32 + (1 << (SHIFT - 1))) >> SHIFT
h_scale = (1 << SHIFT) * W1_scale

""" Квантизация второго слоя """
W2_scale = np.max(np.abs(W2)) / 127.0
W2_int = np.round(W2 / W2_scale).astype(np.int8)
b2_int = np.round(b2 / (h_scale * W2_scale)).astype(np.int32)

""" Integer inference второго слоя """
z2_int = (h_int.astype(np.int32) @ W2_int.astype(np.int32) + b2_int)

""" Проверка """
pred_int = np.argmax(z2_int, axis=1)
accuracy_int = np.mean(pred_int == Y_test)

print("Float accuracy:", accuracy)
print("INT8 SHIFT accuracy:", accuracy_int)

In [ ]:
from pathlib import Path
import os
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


# ============================================================
# Project directories
# ============================================================

VIVADO_FOLDER = os.getenv("VIVADO_FOLDER")

if VIVADO_FOLDER is None:
    raise RuntimeError("Environment variable VIVADO_FOLDER is not set")

PROJECT_DIR = Path(VIVADO_FOLDER) / "Neural_Processor"

SOURCES_DIR = PROJECT_DIR / "files" / "sources"
SIM_DIR = PROJECT_DIR / "files" / "simulations"

SOURCES_DIR.mkdir(parents=True, exist_ok=True)
SIM_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Dataset
# ============================================================

digits = load_digits()

X = digits.data.astype(np.uint8)
Y = digits.target.astype(np.uint8)

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

assert len(X_test) == 360
assert len(Y_test) == 360


# ============================================================
# MEM helpers
# ============================================================

def write_int8_mem(filename, data):
    with open(filename, "w") as file:
        for value in data:
            file.write(f"{int(value) & 0xFF:02X}\n")


def write_int32_mem(filename, value):
    with open(filename, "w") as file:
        file.write(f"{int(value) & 0xFFFFFFFF:08X}\n")


# ============================================================
# Neural network weights
#
# files/sources/
# ============================================================

for neuron in range(8):

    write_int8_mem(
        SOURCES_DIR / f"W1_{neuron}.mem",
        W1_int[:, neuron]
    )

    write_int32_mem(
        SOURCES_DIR / f"B1_{neuron}.mem",
        b1_int[neuron]
    )


for neuron in range(10):

    write_int8_mem(
        SOURCES_DIR / f"W2_{neuron}.mem",
        W2_int[:, neuron]
    )

    write_int32_mem(
        SOURCES_DIR / f"B2_{neuron}.mem",
        b2_int[neuron]
    )


# ============================================================
# Test dataset
#
# 360 images
# files/simulations/
# ============================================================

with open(SIM_DIR / "digits_x.mem", "w") as file:

    for image in X_test:
        for pixel in image:
            file.write(f"{int(pixel):02X}\n")


with open(SIM_DIR / "digits_y.mem", "w") as file:

    for label in Y_test:
        file.write(f"{int(label):X}\n")


# ============================================================
# Info
# ============================================================

print("Generated neural network files:")
print(f"  {SOURCES_DIR}")

print()
print("Generated simulation files:")
print(f"  {SIM_DIR}")

print()
print(f"Test images: {len(X_test)}")
print(f"Test pixels: {X_test.size}")